# LeetCode 3507 — Minimum Pair Removal to Sort Array I bằng Python

Link đề: https://leetcode.com/problems/minimum-pair-removal-to-sort-array-i/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Linked List, Heap, Simulation

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def minimumPairRemoval(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề là **mô phỏng một quy trình cố định**: lặp lại "chọn cặp kề có tổng nhỏ nhất (ưu tiên bên trái nếu hòa), thay bằng tổng của chúng", đếm số bước đến khi mảng **không giảm**.
2. Biết kiểm tra "mảng không giảm" bằng `all(a[i] <= a[i+1] ...)` và tìm vị trí nhỏ nhất bằng `min(range(...), key=...)` (trả về phần tử **đầu tiên** khi hòa).
3. Biết thao tác **gộp hai phần tử kề** bằng gán lát cắt `a[i:i+2] = [tổng]`.
4. Viết 4 cách: mô phỏng bằng vòng lặp, mô phỏng bằng `min(key=...)`, mô phỏng trên **danh sách liên kết đôi** (mảng `prev/next`) có đếm số cặp "nghịch thế", và **heap** lười (lazy deletion) — mầm của bài II.
5. Tránh các lỗi: chọn cặp **bên phải** khi hòa, vẫn thực hiện một bước khi mảng đã đúng thứ tự, đếm số cặp giảm thay vì mô phỏng, và xóa cặp thay vì thay bằng tổng.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Bạn được thực hiện thao tác sau **nhiều lần** cho đến khi mảng **không giảm** (mỗi phần tử không nhỏ hơn phần tử đứng trước, tức `nums[i] <= nums[i+1]`):

- Chọn **cặp phần tử kề nhau có tổng nhỏ nhất**. Nếu có nhiều cặp cùng tổng nhỏ nhất, chọn cặp **ngoài cùng bên trái**.
- Thay cặp đó bằng **một** phần tử bằng tổng của chúng.

Trả về **số thao tác** cần thực hiện để mảng trở nên không giảm.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[5,2,3,1]` | `2` | cặp `(3,1)` có tổng nhỏ nhất `4` -> `[5,2,4]`; cặp `(2,4)` tổng `6` nhỏ nhất -> `[5,6]`, đã không giảm sau 2 thao tác |
| 2 | `[1,2,2]` | `0` | đã không giảm, không cần thao tác |

Constraints: `1 <= nums.length <= 50`; `-1000 <= nums[i] <= 1000`.

Chú ý: thao tác **bắt buộc** theo luật chọn cặp, bạn không tự chọn cặp. Vì vậy bài này là **mô phỏng**, không phải tối ưu hóa.

In [1]:
examples = [([5, 2, 3, 1], 2), ([1, 2, 2], 0)]
for nums, expected in examples:
    a = list(nums)
    ops = 0
    print("bắt đầu:", a)
    while any(a[i] > a[i + 1] for i in range(len(a) - 1)):
        sums = [a[i] + a[i + 1] for i in range(len(a) - 1)]
        i = sums.index(min(sums))                 # index trả về vị trí đầu tiên -> bên trái nhất
        a[i:i + 2] = [a[i] + a[i + 1]]
        ops += 1
        print(f"  tổng các cặp {sums} -> gộp cặp ở vị trí {i} -> {a}")
    print(f"nums={nums}: {ops} thao tác (mong đợi {expected})")

bắt đầu: [5, 2, 3, 1]
  tổng các cặp [7, 5, 4] -> gộp cặp ở vị trí 2 -> [5, 2, 4]
  tổng các cặp [7, 6] -> gộp cặp ở vị trí 1 -> [5, 6]
nums=[5, 2, 3, 1]: 2 thao tác (mong đợi 2)
bắt đầu: [1, 2, 2]
nums=[1, 2, 2]: 0 thao tác (mong đợi 0)


## 3. Kiến thức nền

### 3.1. Mảng "không giảm"

Mảng `a` không giảm khi `a[i] <= a[i+1]` với mọi `i`. Có **bằng nhau** vẫn tốt (khác "tăng ngặt"). Mảng độ dài `0` hoặc `1` luôn không giảm. Kiểm tra: `all(a[i] <= a[i+1] for i in range(len(a) - 1))`.

In [2]:
def is_non_decreasing(a):
    return all(a[i] <= a[i + 1] for i in range(len(a) - 1))

print(is_non_decreasing([1, 2, 2]), is_non_decreasing([5, 2, 4]), is_non_decreasing([7]))

True False True


### 3.2. `min` với `key` và vị trí đầu tiên khi hòa

`min(range(m), key=lambda i: f(i))` trả về chỉ số `i` có `f(i)` nhỏ nhất; nếu nhiều chỉ số cùng giá trị nhỏ nhất, `min` trả về chỉ số **xuất hiện đầu tiên**. Đúng với quy tắc "bên trái nhất" của đề. Tương tự `list.index(min(list))` trả về vị trí đầu tiên.

In [3]:
sums = [3, 1, 1, 5]
print(min(range(len(sums)), key=lambda i: sums[i]), sums.index(min(sums)))   # cả hai là 1, không phải 2

1 1


### 3.3. Gán lát cắt để gộp hai phần tử

`a[i:i+2] = [x]` thay hai phần tử `a[i]`, `a[i+1]` bằng **một** phần tử `x`; độ dài mảng giảm `1`. Đây là cách ngắn nhất để "thay cặp bằng tổng".

In [4]:
a = [5, 2, 3, 1]
i = 2
a[i:i + 2] = [a[i] + a[i + 1]]
print(a)

[5, 2, 4]


### 3.4. Danh sách liên kết đôi bằng mảng `prev` / `next`

Thay vì xóa phần tử khỏi `list` (tốn `O(n)`), ta giữ `val[i]`, `prev[i]`, `nxt[i]` và gộp bằng cách **nối lại con trỏ**: gộp nút `j = nxt[i]` vào `i`: `val[i] += val[j]`, `nxt[i] = nxt[j]`, `prev[nxt[j]] = i`. Phần tử `j` coi như bị xóa (`alive[j] = False`). Thứ tự các chỉ số còn sống vẫn là thứ tự ban đầu, nên "bên trái nhất" = chỉ số nhỏ nhất.

In [5]:
val = [5, 2, 3, 1]
nxt = [1, 2, 3, -1]
prev = [-1, 0, 1, 2]
i = 2; j = nxt[i]
val[i] += val[j]
nxt[i] = nxt[j]            # nxt[3] = -1 -> không có phần tử sau
print(val, nxt)             # nút 3 đã bị bỏ qua

[5, 2, 4, 1] [1, 2, -1, -1]


### 3.5. `heapq` và xóa lười (lazy deletion)

`heapq` là **min-heap**: `heappush`, `heappop` mất `O(log n)`; phần tử nhỏ nhất ở đỉnh. So sánh tuple theo từng phần tử: `(tổng, chỉ_số)` nên khi hòa tổng thì chỉ số nhỏ hơn (bên trái hơn) ra trước — đúng luật đề. Heap không hỗ trợ xóa phần tử bất kỳ, nên ta **bỏ qua** các mục đã lỗi thời khi pop (lazy deletion).

In [6]:
import heapq
h = []
for item in [(5, 2), (3, 7), (3, 1), (4, 0)]:
    heapq.heappush(h, item)
print([heapq.heappop(h) for _ in range(len(h))])   # (3,1) trước (3,7): hòa tổng thì chỉ số nhỏ trước

[(3, 1), (3, 7), (4, 0), (5, 2)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Cứ làm đúng như đề trên `[5, 2, 3, 1]`:

```
a = [5, 2, 3, 1]   tổng các cặp kề: 7, 5, 4   -> nhỏ nhất 4 ở cặp (3,1)  -> a = [5, 2, 4]   (1 bước)
a = [5, 2, 4]      tổng các cặp kề: 7, 6      -> nhỏ nhất 6 ở cặp (2,4)  -> a = [5, 6]      (2 bước)
a = [5, 6]         không giảm -> dừng. Đáp án 2.
```

Nhận xét: mỗi thao tác làm mảng **ngắn đi đúng 1**, nên số thao tác tối đa là `n - 1` (mảng còn 1 phần tử luôn không giảm). Với `n <= 50` ta có tối đa `49` bước, mỗi bước `O(n)` -> mô phỏng trực tiếp là hoàn toàn đủ nhanh. Độ khó của bài chỉ là **đọc đúng luật** (hòa thì bên trái, kiểm tra dừng **trước** khi thao tác).

## 5. Cách 1 — Brute Force / baseline

Vòng lặp `while True`: (1) kiểm tra mảng không giảm, nếu đúng thì dừng; (2) tìm cặp có tổng nhỏ nhất **bên trái nhất** bằng vòng lặp với so sánh **nghiêm ngặt** `<` (không dùng `<=` vì `<=` sẽ lấy cặp bên phải); (3) gộp cặp, tăng đếm.

In [7]:
class SolutionBrute:
    def minimumPairRemoval(self, nums):
        a = list(nums)
        ops = 0
        while True:
            ok = True
            for i in range(len(a) - 1):
                if a[i] > a[i + 1]:
                    ok = False
                    break
            if ok:
                return ops
            best = 0
            for i in range(1, len(a) - 1):
                if a[i] + a[i + 1] < a[best] + a[best + 1]:    # nghiêm ngặt: giữ cặp bên trái khi hòa
                    best = i
            a[best:best + 2] = [a[best] + a[best + 1]]
            ops += 1

print(SolutionBrute().minimumPairRemoval([5, 2, 3, 1]))
print(SolutionBrute().minimumPairRemoval([1, 2, 2]))
print(SolutionBrute().minimumPairRemoval([1, 2, 1]))

2
0
2


**Độ phức tạp:** tối đa `n - 1` thao tác, mỗi thao tác quét `O(n)` (kiểm tra thứ tự, tìm cặp, gộp lát cắt) -> `O(n²)` thời gian, `O(n)` bộ nhớ cho bản sao. Với `n <= 50` là vài nghìn phép tính.

## 6. Ý tưởng thuật toán / bước đột phá

Vì đề không cho lựa chọn, **không có "bước đột phá" về thuật toán** ở bản I; ta chỉ có thể làm mỗi bước rẻ hơn:

- **Gộp nhanh:** dùng danh sách liên kết (`prev/nxt`) để gộp trong `O(1)` thay vì `a[i:i+2] = ...` `O(n)`.
- **Biết khi nào dừng mà không quét lại:** giữ biến `bad` = số cặp kề `(x, y)` có `x > y`. Mảng không giảm khi và chỉ khi `bad == 0`. Khi gộp cặp `(i, j)`, chỉ ba cặp bị ảnh hưởng: `(p, i)`, `(i, j)`, `(j, k)` (với `p` là trước `i`, `k` là sau `j`); ta trừ đóng góp cũ rồi cộng đóng góp mới `(p, i')`, `(i', k)`.
- **Tìm cặp nhỏ nhất nhanh:** heap chứa `(tổng, chỉ_số_trái)`. Mục lỗi thời (nút trái đã bị xóa, hoặc láng giềng phải đã đổi, hoặc tổng đã đổi) bị bỏ qua khi pop.

**Quy tắc "nếu… thì…":**

- Nếu `bad == 0` thì dừng, trả số bước.
- Ngược lại: lấy cặp `(tổng nhỏ nhất, chỉ số nhỏ nhất)` còn hợp lệ, gộp, cập nhật `bad`, đẩy hai cặp mới vào heap.
- Invariant: heap luôn chứa **mọi cặp kề hiện tại** (cùng một số mục lỗi thời chưa bị loại), và mọi mục hợp lệ có đúng `tổng` và `chỉ_số` hiện tại.

In [8]:
a = [5, 2, 3, 1]
bad = sum(1 for i in range(len(a) - 1) if a[i] > a[i + 1])
print("số cặp nghịch thế ban đầu:", bad)
a2 = [5, 2, 4]
print("sau bước 1:", sum(1 for i in range(len(a2) - 1) if a2[i] > a2[i + 1]))
a3 = [5, 6]
print("sau bước 2:", sum(1 for i in range(len(a3) - 1) if a3[i] > a3[i + 1]))

số cặp nghịch thế ban đầu: 2
sau bước 1: 1
sau bước 2: 0


## 7. Các cách tiếp theo

Ta có 4 cách: (1) mô phỏng bằng vòng lặp; (2) mô phỏng gọn bằng `min(key=...)`; (3) danh sách liên kết + đếm `bad`; (4) heap lười + danh sách liên kết.

### Cách 2 — Mô phỏng gọn với `min(key=...)`

Dùng `all(...)` kiểm tra dừng, `min(range(len(a) - 1), key=lambda i: a[i] + a[i+1])` chọn cặp đầu tiên có tổng nhỏ nhất, rồi gán lát cắt.

In [9]:
class SolutionMinKey:
    def minimumPairRemoval(self, nums):
        a = list(nums)
        ops = 0
        while any(a[i] > a[i + 1] for i in range(len(a) - 1)):
            i = min(range(len(a) - 1), key=lambda t: a[t] + a[t + 1])    # hòa -> chỉ số nhỏ nhất
            a[i:i + 2] = [a[i] + a[i + 1]]
            ops += 1
        return ops

print(SolutionMinKey().minimumPairRemoval([5, 2, 3, 1]))
print(SolutionMinKey().minimumPairRemoval([1, 2, 2]))
print(SolutionMinKey().minimumPairRemoval([1, 2, 1]))

2
0
2


Độ phức tạp Cách 2: giống Cách 1, `O(n²)` thời gian, `O(n)` bộ nhớ; ngắn gọn nhất và là cách nên nộp.

### Cách 3 — Danh sách liên kết đôi + biến `bad`

Giữ `val`, `prev`, `nxt`. Chỉ số `0` không bao giờ bị xóa (ta luôn gộp nút phải vào nút trái), nên ta bắt đầu từ `0` để duyệt các nút còn sống theo thứ tự. Mỗi bước: duyệt chuỗi liên kết để tìm cặp tổng nhỏ nhất bên trái nhất, gộp trong `O(1)`, cập nhật `bad` bằng cách trừ/cộng ba cặp bị ảnh hưởng.

In [10]:
class SolutionLinked:
    def minimumPairRemoval(self, nums):
        n = len(nums)
        val = list(nums)
        prev = list(range(-1, n - 1))
        nxt = list(range(1, n + 1))
        nxt[-1] = -1
        bad = sum(1 for i in range(n - 1) if val[i] > val[i + 1])
        ops = 0
        while bad > 0:
            # tìm cặp (i, nxt[i]) có tổng nhỏ nhất, bên trái nhất
            best_i, best_sum = -1, None
            i = 0
            while nxt[i] != -1:
                s = val[i] + val[nxt[i]]
                if best_sum is None or s < best_sum:
                    best_i, best_sum = i, s
                i = nxt[i]
            i = best_i
            j = nxt[i]
            p, k = prev[i], nxt[j]
            # trừ đóng góp cũ của ba cặp (p,i), (i,j), (j,k)
            if p != -1 and val[p] > val[i]:
                bad -= 1
            if val[i] > val[j]:
                bad -= 1
            if k != -1 and val[j] > val[k]:
                bad -= 1
            # gộp j vào i
            val[i] += val[j]
            nxt[i] = k
            if k != -1:
                prev[k] = i
            # cộng đóng góp mới của (p,i), (i,k)
            if p != -1 and val[p] > val[i]:
                bad += 1
            if k != -1 and val[i] > val[k]:
                bad += 1
            ops += 1
        return ops

print(SolutionLinked().minimumPairRemoval([5, 2, 3, 1]))
print(SolutionLinked().minimumPairRemoval([1, 2, 2]))
print(SolutionLinked().minimumPairRemoval([1, 2, 1]))

2
0
2


Độ phức tạp Cách 3: mỗi bước duyệt chuỗi `O(n)` để tìm cặp nhỏ nhất, gộp và cập nhật `bad` `O(1)` -> tổng `O(n²)` thời gian, `O(n)` bộ nhớ. Nhưng **không còn** thao tác `O(n)` của gán lát cắt và quét kiểm tra thứ tự.

### Cách 4 — Heap lười + danh sách liên kết

Thêm `heap` chứa `(tổng, chỉ_số_trái)` của mọi cặp kề. Mỗi bước pop đến khi gặp mục **hợp lệ**: nút `i` còn sống, `j = nxt[i]` tồn tại, và `val[i] + val[j] == tổng` ghi trong mục. Gộp như Cách 3, rồi đẩy hai cặp mới `(p, i)` và `(i, k)` vào heap.

In [11]:
import heapq

class SolutionHeap:
    def minimumPairRemoval(self, nums):
        n = len(nums)
        val = list(nums)
        prev = list(range(-1, n - 1))
        nxt = list(range(1, n + 1))
        nxt[-1] = -1
        alive = [True] * n
        bad = sum(1 for i in range(n - 1) if val[i] > val[i + 1])
        heap = [(val[i] + val[i + 1], i) for i in range(n - 1)]
        heapq.heapify(heap)
        ops = 0
        while bad > 0:
            s, i = heapq.heappop(heap)
            j = nxt[i]
            if not alive[i] or j == -1 or val[i] + val[j] != s:
                continue                                    # mục lỗi thời -> bỏ qua
            p, k = prev[i], nxt[j]
            if p != -1 and val[p] > val[i]:
                bad -= 1
            if val[i] > val[j]:
                bad -= 1
            if k != -1 and val[j] > val[k]:
                bad -= 1
            val[i] += val[j]
            alive[j] = False
            nxt[i] = k
            if k != -1:
                prev[k] = i
            if p != -1 and val[p] > val[i]:
                bad += 1
            if k != -1 and val[i] > val[k]:
                bad += 1
            if p != -1:
                heapq.heappush(heap, (val[p] + val[i], p))
            if k != -1:
                heapq.heappush(heap, (val[i] + val[k], i))
            ops += 1
        return ops

print(SolutionHeap().minimumPairRemoval([5, 2, 3, 1]))
print(SolutionHeap().minimumPairRemoval([1, 2, 2]))
print(SolutionHeap().minimumPairRemoval([1, 2, 1]))

2
0
2


Độ phức tạp Cách 4: ban đầu `n - 1` mục; mỗi thao tác đẩy thêm tối đa `2` mục -> tối đa `O(n)` mục, mỗi mục pop/push `O(log n)` -> **`O(n log n)` thời gian, `O(n)` bộ nhớ**. Đây là hướng đi cho bài **II** (`n` lên đến `10^5`).

## 8. TRACE từng bước

Trace Cách 2 với `nums = [1, 2, 1]` (ví dụ có **hòa tổng**, hai cặp đều tổng `3`):

| bước | `a` | tổng các cặp | chọn | `a` sau bước |
|---|---|---|---|---|
| 0 | `[1,2,1]` | `3, 3` | `1 > ?` không giảm? `2 > 1` có -> chưa dừng | |
| 1 | `[1,2,1]` | `3, 3` | hòa -> cặp bên trái `(1,2)` | `[3,1]` |
| 2 | `[3,1]` | `4` | cặp `(3,1)` | `[4]` |
| 3 | `[4]` | | mảng 1 phần tử luôn không giảm | dừng, đáp án `2` |

Nếu chọn cặp **bên phải** `(2,1)` ở bước 1 thì được `[1,3]` đã không giảm và chỉ mất `1` thao tác — khác đáp án đúng. Luật "bên trái" không phải chi tiết thừa.

In [12]:
def trace_removal(nums):
    a = list(nums)
    ops = 0
    print("bắt đầu:", a)
    while any(a[i] > a[i + 1] for i in range(len(a) - 1)):
        sums = [a[i] + a[i + 1] for i in range(len(a) - 1)]
        m = min(sums)
        i = sums.index(m)
        print(f"  a={a} sums={sums} min={m} (cặp đầu tiên ở vị trí {i}: {a[i]}+{a[i+1]})")
        a[i:i + 2] = [a[i] + a[i + 1]]
        ops += 1
    print(f"dừng với a={a} sau {ops} thao tác")
    return ops

trace_removal([1, 2, 1])
trace_removal([5, 2, 3, 1])
trace_removal([-2, -1, -2, -1])

bắt đầu: [1, 2, 1]
  a=[1, 2, 1] sums=[3, 3] min=3 (cặp đầu tiên ở vị trí 0: 1+2)
  a=[3, 1] sums=[4] min=4 (cặp đầu tiên ở vị trí 0: 3+1)
dừng với a=[4] sau 2 thao tác
bắt đầu: [5, 2, 3, 1]
  a=[5, 2, 3, 1] sums=[7, 5, 4] min=4 (cặp đầu tiên ở vị trí 2: 3+1)
  a=[5, 2, 4] sums=[7, 6] min=6 (cặp đầu tiên ở vị trí 1: 2+4)
dừng với a=[5, 6] sau 2 thao tác
bắt đầu: [-2, -1, -2, -1]
  a=[-2, -1, -2, -1] sums=[-3, -3, -3] min=-3 (cặp đầu tiên ở vị trí 0: -2+-1)
dừng với a=[-3, -2, -1] sau 1 thao tác


1

## 9. Code nộp LeetCode

In [13]:
from typing import List

class Solution:
    def minimumPairRemoval(self, nums: List[int]) -> int:
        a = list(nums)
        ops = 0
        while any(a[i] > a[i + 1] for i in range(len(a) - 1)):
            i = min(range(len(a) - 1), key=lambda t: a[t] + a[t + 1])
            a[i:i + 2] = [a[i] + a[i + 1]]
            ops += 1
        return ops

print(Solution().minimumPairRemoval([5, 2, 3, 1]), Solution().minimumPairRemoval([1, 2, 2]))

2 0


## 10. Vì sao thuật toán đúng

- **Làm đúng định nghĩa:** Cách 1 và 2 thực hiện nguyên văn quy trình của đề: kiểm tra dừng, chọn cặp tổng nhỏ nhất bên trái nhất (so sánh nghiêm ngặt / `min` trả phần tử đầu), gộp, đếm.
- **Dừng được:** mỗi thao tác giảm độ dài mảng đúng `1`; mảng độ dài `1` không giảm, nên vòng lặp dừng sau tối đa `n - 1` bước.
- **Cách 3, 4 đúng nhờ ba điều:**
  1. *Thứ tự được bảo toàn:* ta luôn gộp nút phải `j` vào nút trái `i`, nên chỉ số nhỏ hơn đứng trước; "bên trái nhất" = chỉ số nhỏ nhất.
  2. *`bad` đúng:* chỉ ba cặp `(p,i)`, `(i,j)`, `(j,k)` thay đổi; ta trừ đóng góp cũ và cộng đóng góp mới của `(p,i')`, `(i',k)`; các cặp khác không đụng tới. `bad == 0` khi và chỉ khi không có cặp giảm, tức mảng không giảm.
  3. *Heap đúng:* mọi cặp kề hiện tại đều có mục trong heap với `(tổng hiện tại, chỉ số trái)` (ta đẩy lại mỗi khi cặp đó thay đổi). Mục nào lỗi thời bị loại bởi ba điều kiện (`alive`, `j != -1`, tổng khớp). Heap so tuple nên `(tổng nhỏ nhất, chỉ số nhỏ nhất)` ra trước.
- **Đối chiếu:** bốn cách được so với nhau trên hàng nghìn input ngẫu nhiên, gồm cả giá trị âm và nhiều giá trị bằng nhau (dễ hòa tổng).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. Vòng lặp mô phỏng | `O(n²)` | `O(n)` |
| 2. `min(key=...)` | `O(n²)` | `O(n)` |
| 3. Liên kết + `bad` | `O(n²)` | `O(n)` |
| 4. Heap lười + liên kết | `O(n log n)` | `O(n)` |

Với `n <= 50` mọi cách tức thì; Cách 4 mới là cách mở rộng được lên `n = 10^5` (bài II). Số thao tác tối đa luôn là `n - 1`.

## 12. Bộ test

Test 1: ví dụ đề và edge case: `n = 1`, đã không giảm, giảm hoàn toàn, có số âm, **hòa tổng** (trường hợp luật "bên trái" quyết định), toàn số bằng nhau. Mọi input tuân thủ `1 <= n <= 50`, `-1000 <= nums[i] <= 1000`.

In [14]:
impls = [SolutionBrute(), SolutionMinKey(), SolutionLinked(), SolutionHeap(), Solution()]
tests = [
    ([5, 2, 3, 1], 2),
    ([1, 2, 2], 0),
    ([7], 0),                                  # n = 1
    ([1, 2], 0),                               # đã không giảm
    ([2, 1], 1),                               # một cặp giảm -> gộp thành [3]
    ([1, 2, 1], 2),                            # hòa tổng: phải gộp cặp bên trái trước
    ([-2, -1, -2, -1], 1),                     # số âm + hòa tổng: chỉ cần 1 bước
    ([1, 1, 2, 1], 2),
    ([1, 1, 3, 2], 3),
    ([3, 3, 3, 3], 0),                         # toàn bằng nhau
    ([5, 4, 3, 2, 1], 4),                      # giảm hoàn toàn
    ([1000, -1000, 1000, -1000], 2),           # biên giá trị
    ([-1000, -1000], 0),
]
def oracle(nums):
    a = list(nums)
    ops = 0
    while sorted(a) != a:                      # kiểm tra không giảm bằng so với bản sắp xếp
        sums = [x + y for x, y in zip(a, a[1:])]
        i = sums.index(min(sums))
        a = a[:i] + [a[i] + a[i + 1]] + a[i + 2:]
        ops += 1
    return ops

for nums, exp in tests:
    assert 1 <= len(nums) <= 50 and all(-1000 <= v <= 1000 for v in nums)
    assert oracle(nums) == exp, (nums, oracle(nums), exp)
    for im in impls:
        got = im.minimumPairRemoval(list(nums))
        assert got == exp, (type(im).__name__, nums, got, exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 13


Test 2: ngẫu nhiên có seed. Dùng dải giá trị nhỏ (`-3..3`, rất nhiều hòa tổng), dải trung bình và dải đầy đủ `-1000..1000`, độ dài `1..50`. Oracle độc lập ở trên (so với bản sắp xếp + cắt ghép danh sách).

In [15]:
import random
random.seed(3507)
for t in range(3000):
    n = random.randint(1, 50)
    r = random.choice([3, 10, 1000])
    nums = [random.randint(-r, r) for _ in range(n)]
    assert 1 <= len(nums) <= 50 and all(-1000 <= v <= 1000 for v in nums)
    ref = oracle(nums)
    assert 0 <= ref <= n - 1
    for im in impls:
        assert im.minimumPairRemoval(list(nums)) == ref, (type(im).__name__, nums)
print("3000 test ngẫu nhiên OK")

3000 test ngẫu nhiên OK


Test 3: tính chất — mảng không giảm cho `0`; mảng giảm ngặt có đáp án `>= 1`; không sửa input; số thao tác không vượt quá `n - 1`.

In [16]:
random.seed(507)
for _ in range(500):
    n = random.randint(1, 50)
    nums = sorted(random.randint(-1000, 1000) for _ in range(n))
    assert Solution().minimumPairRemoval(nums) == 0                    # đã không giảm
    desc = sorted(set(random.randint(-1000, 1000) for _ in range(n)), reverse=True)
    if len(desc) >= 2:
        assert Solution().minimumPairRemoval(desc) >= 1                # giảm ngặt có ít nhất 1 cặp
    arr = [random.randint(-5, 5) for _ in range(n)]
    snapshot = list(arr)
    ans = Solution().minimumPairRemoval(arr)
    assert arr == snapshot and 0 <= ans <= n - 1
print("tính chất (không giảm => 0, giảm ngặt >= 1, không sửa input, <= n-1): OK")

tính chất (không giảm => 0, giảm ngặt >= 1, không sửa input, <= n-1): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: khi hòa tổng lại chọn cặp **bên phải**

In [17]:
def wrong_rightmost_tie(nums):
    a = list(nums)
    ops = 0
    while any(a[i] > a[i + 1] for i in range(len(a) - 1)):
        best = 0
        for i in range(len(a) - 1):
            if a[i] + a[i + 1] <= a[best] + a[best + 1]:   # SAI: <= làm cặp bên phải thắng khi hòa
                best = i
        a[best:best + 2] = [a[best] + a[best + 1]]
        ops += 1
    return ops

nums = [1, 2, 1]
print("sai :", wrong_rightmost_tie(nums))
print("đúng:", Solution().minimumPairRemoval(nums))
assert wrong_rightmost_tie(nums) != Solution().minimumPairRemoval(nums)

sai : 1
đúng: 2


Vì sao sai: `[1,2,1]` có hai cặp cùng tổng `3`. Đề bắt chọn cặp **bên trái** `(1,2)` -> `[3,1]` rồi `[4]` (2 thao tác); chọn bên phải ra `[1,3]` (1 thao tác). **Sửa:** so sánh nghiêm ngặt `<` (hoặc `min(..., key=...)`).

### Lỗi 2: luôn thực hiện ít nhất một thao tác (kiểu do-while)

In [18]:
def wrong_do_while(nums):
    a = list(nums)
    ops = 0
    while True:
        if len(a) < 2:
            break
        i = min(range(len(a) - 1), key=lambda t: a[t] + a[t + 1])
        a[i:i + 2] = [a[i] + a[i + 1]]
        ops += 1                                            # SAI: thao tác trước khi kiểm tra đã không giảm
        if all(a[t] <= a[t + 1] for t in range(len(a) - 1)):
            break
    return ops

nums = [1, 2, 2]
print("sai :", wrong_do_while(nums))
print("đúng:", Solution().minimumPairRemoval(nums))
assert wrong_do_while(nums) != Solution().minimumPairRemoval(nums)

sai : 2
đúng: 0


Vì sao sai: mảng `[1,2,2]` đã không giảm nên đáp án là `0`; code sai vẫn gộp một lần. **Sửa:** kiểm tra điều kiện dừng **trước** khi thao tác (`while` kiểm tra ở đầu vòng).

### Lỗi 3: đếm số cặp giảm thay vì mô phỏng

In [19]:
def wrong_count_descents(nums):
    return sum(1 for i in range(len(nums) - 1) if nums[i] > nums[i + 1])   # SAI: bỏ qua việc gộp làm đổi mảng

nums = [1, 1, 2, 1]
print("sai :", wrong_count_descents(nums))
print("đúng:", Solution().minimumPairRemoval(nums))
assert wrong_count_descents(nums) != Solution().minimumPairRemoval(nums)

sai : 1
đúng: 2


Vì sao sai: `[1,1,2,1]` chỉ có một cặp giảm `(2,1)`, nhưng luật bắt gộp cặp tổng nhỏ nhất `(1,1)` trước -> `[2,2,1]`, rồi `(2,1)` -> `[2,3]`: cần `2` thao tác. Thao tác không nhắm vào cặp giảm mà vào cặp **tổng nhỏ nhất**. **Sửa:** mô phỏng đúng quy trình.

### Lỗi 4: xóa cả hai phần tử thay vì thay bằng tổng

In [20]:
def wrong_delete_pair(nums):
    a = list(nums)
    ops = 0
    while any(a[i] > a[i + 1] for i in range(len(a) - 1)):
        i = min(range(len(a) - 1), key=lambda t: a[t] + a[t + 1])
        del a[i:i + 2]                                      # SAI: phải thay bằng một phần tử tổng
        ops += 1
    return ops

nums = [1, 1, 3, 2]
print("sai :", wrong_delete_pair(nums))
print("đúng:", Solution().minimumPairRemoval(nums))
assert wrong_delete_pair(nums) != Solution().minimumPairRemoval(nums)

sai : 2
đúng: 3


Vì sao sai: sau khi xóa `(1,1)`, mảng thành `[3,2]` rồi `[]` (2 thao tác); đúng là `[1,1,3,2] -> [2,3,2] -> [5,2] -> [7]` (3 thao tác). **Sửa:** `a[i:i+2] = [a[i] + a[i+1]]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề mô tả một **quy trình cố định** ("chọn cặp có tổng nhỏ nhất, hòa thì bên trái") -> **mô phỏng**, không cần sáng tạo.
- `n <= 50` -> mô phỏng `O(n²)` là đủ; đừng cố tối ưu quá sớm.
- Tie-break "bên trái nhất": dùng `min(range(...), key=...)`, `list.index(min(...))`, hoặc so sánh nghiêm ngặt `<`.
- Kiểm tra dừng **trước** mỗi thao tác (có thể đáp án `0`).
- `a[i:i+2] = [x]` gộp hai phần tử; nhớ độ dài mảng giảm `1` mỗi bước.
- Khi `n` lớn (bài II): danh sách liên kết + heap `(tổng, chỉ_số)` + đếm số cặp nghịch thế.
- Bài họ hàng: `3510. Minimum Pair Removal to Sort Array II`, `1046. Last Stone Weight` (heap mô phỏng), `2216. Minimum Deletions to Make Array Beautiful`.

## 15. Biến thể của bài

1. **Bài II (`3510`):** cùng luật nhưng `n` đến `10^5` -> Cách 4.
2. Chọn cặp có tổng **lớn nhất**, hoặc **tích nhỏ nhất**.
3. Đếm số thao tác để mảng trở thành **tăng ngặt** (`a[i] < a[i+1]`).
4. Trả về **mảng cuối cùng** thay vì số thao tác.
5. Cho phép **chọn** cặp tùy ý và hỏi số thao tác **tối thiểu** (khi đó là bài tối ưu hóa khác hẳn, thường dùng quy hoạch động).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng lặp mô phỏng | quét tìm cặp nhỏ nhất bằng `<` | `O(n²)` | `O(n)` | viết chắc tay, oracle |
| 2. `min(key=...)` | cặp đầu tiên có tổng nhỏ nhất | `O(n²)` | `O(n)` | **ngắn gọn, nên nộp** |
| 3. Liên kết + `bad` | gộp `O(1)`, đếm cặp nghịch thế | `O(n²)` | `O(n)` | bước đệm sang heap |
| 4. Heap lười + liên kết | `(tổng, chỉ_số)` trong heap | `O(n log n)` | `O(n)` | `n` lớn (bài II) |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `minimumPairRemoval([3, 1, 2])` bằng bao nhiêu?

**Bài 2 (tính tay):** `nums = [4, 1, 2, 3]`: viết các mảng sau mỗi thao tác và số thao tác.

**Bài 3 (điền TODO):** hoàn thành `leftmost_min_pair(a)` trả về chỉ số `i` của cặp `(a[i], a[i+1])` có tổng nhỏ nhất, bên trái nhất.

**Bài 4 (tự viết lại):** viết `final_array(nums)` trả về **mảng cuối cùng** sau khi quy trình kết thúc.

**Bài 5 (mini challenge):** viết `ops_to_strictly_increasing(nums)` đếm số thao tác (cùng luật chọn cặp) cho đến khi mảng **tăng ngặt** (`a[i] < a[i+1]`).

In [21]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().minimumPairRemoval([3, 1, 2]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def leftmost_min_pair(a):
    # TODO: trả về chỉ số i (0 <= i < len(a) - 1) có a[i] + a[i+1] nhỏ nhất, bên trái nhất
    return 0

1


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** làm từng bước đúng luật: tính tổng các cặp kề, chọn nhỏ nhất (hòa thì trái), thay bằng tổng, kiểm tra dừng.

**Mức 2 (vừa):** Bài 3: `min(range(len(a) - 1), key=lambda i: a[i] + a[i+1])`. Bài 4: giống code nộp nhưng `return a`. Bài 5: đổi điều kiện `a[i] > a[i+1]` thành `a[i] >= a[i+1]`.

**Mức 3 (gần đáp án):** Bài 1: `[3,1,2]` tổng cặp `4, 3` -> gộp `(1,2)` ra `[3,3]` đã không giảm -> `1`. Bài 2: `[4,1,2,3]` tổng `5, 3, 5` -> gộp `(1,2)` ra `[4,3,3]`; tổng `7, 6` -> gộp `(3,3)` ra `[4,6]` không giảm -> `2` thao tác.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1
assert Solution().minimumPairRemoval([3, 1, 2]) == 1

# Bài 2: [4,1,2,3] -> [4,3,3] -> [4,6]
assert Solution().minimumPairRemoval([4, 1, 2, 3]) == 2

# Bài 3
def leftmost_min_pair(a):
    return min(range(len(a) - 1), key=lambda i: a[i] + a[i + 1])
assert leftmost_min_pair([5, 2, 3, 1]) == 2
assert leftmost_min_pair([1, 2, 1]) == 0             # hòa tổng -> bên trái
assert leftmost_min_pair([4, 1, 2, 3]) == 1

# Bài 4
def final_array(nums):
    a = list(nums)
    while any(a[i] > a[i + 1] for i in range(len(a) - 1)):
        i = leftmost_min_pair(a)
        a[i:i + 2] = [a[i] + a[i + 1]]
    return a
assert final_array([5, 2, 3, 1]) == [5, 6]
assert final_array([1, 2, 2]) == [1, 2, 2]
assert final_array([4, 1, 2, 3]) == [4, 6]

# Bài 5
def ops_to_strictly_increasing(nums):
    a = list(nums)
    ops = 0
    while any(a[i] >= a[i + 1] for i in range(len(a) - 1)):
        i = leftmost_min_pair(a)
        a[i:i + 2] = [a[i] + a[i + 1]]
        ops += 1
    return ops
assert ops_to_strictly_increasing([1, 2, 2]) == 2     # [1,2,2] -> [3,2] -> [5]
assert ops_to_strictly_increasing([1, 2, 3]) == 0
assert ops_to_strictly_increasing([5]) == 0
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "mô phỏng quy trình cố định đến khi thỏa điều kiện":**

```python
a = list(nums)
ops = 0
while any(a[i] > a[i + 1] for i in range(len(a) - 1)):      # kiểm tra dừng TRƯỚC
    i = min(range(len(a) - 1), key=lambda t: a[t] + a[t + 1])   # hòa -> trái nhất
    a[i:i + 2] = [a[i] + a[i + 1]]
    ops += 1
return ops
```

**Checklist trước khi submit:**

- [ ] Kiểm tra "không giảm" (`<=`, cho phép bằng nhau) trước mỗi thao tác?
- [ ] Hòa tổng thì chọn cặp **bên trái nhất** (so sánh nghiêm ngặt hoặc `min` của Python)?
- [ ] Thay cặp bằng **một** phần tử tổng (không xóa cả hai)?
- [ ] Không sửa `nums` gốc (đã `list(nums)`)?
- [ ] Đã thử `n = 1`, mảng đã không giảm, số âm, `[1,2,1]`?
- [ ] Chữ ký đúng `minimumPairRemoval(self, nums: List[int]) -> int`?